In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.6 The Kalman Filter: Least Squares, Recursively

In [ ]:
from ecp.style import header, use_style

use_style()
header(
    volume="Chapter VI — Structure, Graphs, and Fast Algorithms",
    number="6.6",
    title="The Kalman Filter: Least Squares, Recursively",
    blurb="Feed least squares its rows one at a time and the update is a "
    "rank-one identity the reader already proved. Add a dynamics matrix and "
    "the same recursion tracks a moving target — the Kalman filter, gated "
    "equal to batch least squares on every prefix, and equal to its own "
    "steady-state Riccati equation at the end.",
    difficulty="advanced",
    estimate="120–150 min",
)

## Notebook overview

[§2.3](../02-orthogonality/least-squares-four-ways.ipynb) solved least
squares with all the rows on the table. This notebook asks the streaming
question: rows arrive one at a time, and re-solving from scratch at every
arrival wastes everything already computed. The answer is **recursive
least squares** — a rank-one update of the normal equations whose inverse
form is exactly the Sherman–Morrison identity the reader proved in
[§1.3](../01-matrices/inverses-rank-cr.ipynb) — and its central promise is
gateable to near rounding: after *every* row, the recursive estimate
equals the batch solution on the rows so far.

Then the structural turn, and the reason this notebook lives in Chapter
VI: give the unknown a **dynamics matrix** — let the state *move* between
measurements — and the same two-line update becomes the **Kalman
filter** {cite}`kalman1960`, the workhorse estimator of navigation,
tracking and control. The kinship is not an analogy: with the dynamics
frozen ($F = I$, $Q = 0$) the filter's output reproduces recursive least
squares to $10^{-12}$, step for step. The filter also diagnoses itself —
its innovation sequence should be white with predicted variance, its gain
should settle to the fixed point of a Riccati equation — and both
predictions are gated against independent routes.

> **How to read a check.** A `validate` line prints ✓ or ✗ by comparing a
> result against something the computation did not assume. A ✗ flags a
> mismatch to investigate, never a verdict on its own.

> **Scope.** Kalman's original paper is {cite}`kalman1960`; Golub and
> Van Loan {cite}`golub2013` §6.5 treats updating least squares, and the
> Sherman–Morrison identity behind the gain is
> [§1.3](../01-matrices/inverses-rank-cr.ipynb)'s Exercise 6.

## Theory in brief

### Least squares, one row at a time

With rows $\mathbf{a}_1^{\top}, \dots, \mathbf{a}_k^{\top}$ stacked in
$A_k$ and measurements $\mathbf{y}_k$, the batch solution solves the
normal equations of
[§2.3](../02-orthogonality/least-squares-four-ways.ipynb):

```{math}
:label: eq-kf-normal
G_k\,\hat{\mathbf{x}}_k = \mathbf{c}_k,
\qquad G_k = A_k^{\top}A_k,\quad \mathbf{c}_k = A_k^{\top}\mathbf{y}_k .
```

A new row touches both sides by a **rank-one** amount:

```{math}
:label: eq-kf-rankone
G_{k+1} = G_k + \mathbf{a}_{k+1}\mathbf{a}_{k+1}^{\top},
\qquad
\mathbf{c}_{k+1} = \mathbf{c}_k + y_{k+1}\,\mathbf{a}_{k+1} ,
```

so the *information* accumulates by outer products. Inverting the update
instead of the matrix is Sherman–Morrison
([§1.3](../01-matrices/inverses-rank-cr.ipynb)): with $P_k = G_k^{-1}$,

```{math}
:label: eq-kf-sm
\mathbf{k} = \frac{P_k\,\mathbf{a}}{1 + \mathbf{a}^{\top}P_k\,\mathbf{a}},
\qquad
\hat{\mathbf{x}} \leftarrow \hat{\mathbf{x}} +
\mathbf{k}\,(y - \mathbf{a}^{\top}\hat{\mathbf{x}}),
\qquad
P \leftarrow P - \mathbf{k}\,\mathbf{a}^{\top}P ,
```

a per-row cost of $O(p^2)$ against the $O(p^3)$ of re-solving — and an
*identity*, not an approximation: {eq}`eq-kf-sm` reproduces the batch
answer exactly, which is precisely what the exercises gate.

### Let the state move

The Kalman filter estimates a state obeying

```{math}
:label: eq-kf-model
\mathbf{x}_{t+1} = F\mathbf{x}_t + \mathbf{w}_t,
\qquad z_t = H\mathbf{x}_t + v_t,
\qquad \mathbf{w} \sim Q,\; v \sim R,
```

by alternating a **prediction** through the dynamics,

```{math}
:label: eq-kf-predict
\mathbf{x} \leftarrow F\mathbf{x}, \qquad P \leftarrow FPF^{\top} + Q ,
```

with an **update** that is {eq}`eq-kf-sm` with the measurement noise $R$
in the denominator:

```{math}
:label: eq-kf-update
S = HPH^{\top} + R,\quad
K = PH^{\top}S^{-1},\quad
\mathbf{x} \leftarrow \mathbf{x} + K\,\nu,\quad
P \leftarrow P - KHP,
```

where $\nu = z - H\mathbf{x}$ is the **innovation** — what the
measurement says that the prediction did not already know. When the
model is right, the innovations are white with variance $S$: the filter
publishes its own error bars, and they are checkable.

### The gain's fixed point

Iterating {eq}`eq-kf-predict`–{eq}`eq-kf-update` drives the predicted
covariance to the fixed point of the discrete algebraic Riccati equation

```{math}
:label: eq-kf-dare
P = F\bigl(P - PH^{\top}(HPH^{\top} + R)^{-1}HP\bigr)F^{\top} + Q ,
```

so the gain settles to a constant $K_{\infty}$ computable *without
filtering anything* — an independent route
(`scipy.linalg.solve_discrete_are`) the running filter is gated against.

---
## Setup

Data only: the streaming least-squares scene (80 rows of a seeded line
fit) and the tracking scene (a constant-velocity target simulated from
{eq}`eq-kf-model` with seeded process and measurement noise). Every
method — the rank-one update, the Sherman–Morrison gain, the Kalman
step — is built in the exercises, where it is the lesson.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_discrete_are

from ecp import validate
from ecp.style import use_style

use_style()
rng = np.random.default_rng(0)  # every random array below comes from this seed

EPS = np.finfo(float).eps

# data: the streaming scene — 80 rows of the line y = 1.5 + 0.8 t at noise
# 0.3, arriving one at a time. Rows of A are (1, t_k).
N_ROWS = 80
t_r = np.linspace(0.0, 4.0, N_ROWS)
A_ROWS = np.column_stack([np.ones(N_ROWS), t_r])
Y_ROWS = 1.5 + 0.8 * t_r + 0.3 * rng.standard_normal(N_ROWS)

# data: the tracking scene — a constant-velocity target (Eq. 4): state
# (position, velocity), dt = 1, white-acceleration process noise at 0.05,
# position measured at noise 0.5, 120 steps from (0, 1).
DT = 1.0
Q_ACC = 0.05
R_MEAS = 0.5
T_STEPS = 120
F_DYN = np.array([[1.0, DT], [0.0, 1.0]])
Q_DYN = Q_ACC**2 * np.array([[DT**3 / 3, DT**2 / 2], [DT**2 / 2, DT]])
H_OBS = np.array([[1.0, 0.0]])
_x = np.array([0.0, 1.0])
TRUTH = np.empty((T_STEPS, 2))
MEAS = np.empty(T_STEPS)
for _t in range(T_STEPS):
    _x = F_DYN @ _x + Q_ACC * np.array([DT**2 / 2, DT]) * rng.standard_normal()
    TRUTH[_t] = _x
    MEAS[_t] = _x[0] + R_MEAS * rng.standard_normal()

## Exercise 1: Recursive least squares, information form

{eq}`eq-kf-rankone` promises that streaming and batch see the same
normal equations — so after *every* row, the recursive estimate must
equal `np.linalg.lstsq` on the prefix. That identity is the exercise.

**Part a)** Initialise exactly: solve {eq}`eq-kf-normal` on the first
two rows of the Setup's `A_ROWS`, `Y_ROWS` ($G_2 = A_2^{\top}A_2$,
$2\times2$ and invertible). Then stream rows $k = 3, \dots, 80$ through
the rank-one update {eq}`eq-kf-rankone`, re-solving the $2\times2$
system at each arrival.

**Write this one yourself** — the update is two `+=` lines and one
solve, and Exercises 2–4 are all refactorings of what you write here.

**Part b)** Gate the identity: at **every** prefix $k$, the streamed
$\hat{\mathbf{x}}_k$ equals `np.linalg.lstsq(A_ROWS[:k], Y_ROWS[:k],
rcond=None)` to $10^{-11}$ relative (78 comparisons, worst reported —
measured near $10^{-15}$: the same equations, met from two directions).

**Part c)** Draw the convergence: the data, the final batch line, and
the streamed line at $k = 4, 10, 20, 80$ — the estimate settling as
evidence accumulates.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.below(
    worst_prefix, 1e-11,
    "after every row, streaming equals batch (Eqs. 1-2)",
    "78 prefix solutions compared against np.linalg.lstsq: the rank-one "
    "update is an identity about the normal equations, not an "
    "approximation to them",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2: The gain form: Sherman–Morrison does the inverting

Exercise 1 re-solved a linear system per row; {eq}`eq-kf-sm` updates the
*inverse* directly — [§1.3](../01-matrices/inverses-rank-cr.ipynb)'s
Sherman–Morrison identity, earning its keep. Per row it costs $O(p^2)$
(three matrix–vector products) against the information form's $O(p^3)$
solve — at $p = 2$ nobody cares, at $p = 10^4$ everything does, and the
count follows from the formulas alone.

**Part a)** Write `rls_gain_step(x, P, a, y)` implementing
{eq}`eq-kf-sm` and returning `(x, P)`. Initialise from the same exact
two-row start ($P_2 = G_2^{-1}$ by `np.linalg.inv`) and stream the same
78 rows.

**Write this one yourself** — the three lines are the Kalman update in
embryo, and Exercise 4 gates that sentence literally.

**Part b)** Gate both identities at every step: the gain-form estimate
equals Exercise 1's information-form estimate to $10^{-12}$ relative,
and $P_k$ equals `np.linalg.inv(G_k)` to $10^{-11}$ relative — the
inverse maintained without ever inverting.

**Part c)** Gate the covariance's health: every $P_k$ symmetric to
$10^{-13}$ and positive definite (smallest eigenvalue positive) — $P$
is the estimate's error covariance, and a covariance that loses
symmetry or positivity has stopped meaning anything.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.below(
    worst_route, 1e-12,
    "the Sherman-Morrison gain and the re-solved system agree, step for "
    "step (Eq. 3)",
    "two refactorings of one identity — the 1.3 formula run forward 78 "
    "times without drifting from the truth it rearranges",
)
validate.below(
    worst_inv, 1e-11,
    "and P tracks the exact inverse without ever inverting",
    "the O(p^2) route maintains what the O(p^3) route recomputes — the "
    "entire point of updating, gated as an identity",
)
validate.check(
    worst_sym < 1e-13 and min_eig_P > 0.0,
    "the covariance stays symmetric and positive definite throughout",
    f"symmetry {worst_sym:.1e}, min eigenvalue {min_eig_P:.2e}: an error "
    "covariance that loses either has stopped meaning anything",
)

## Exercise 3: The Kalman filter, assembled

Add {eq}`eq-kf-predict` in front of the update and the streaming
estimator can chase a *moving* state. The Setup simulated the scene:
a constant-velocity target under {eq}`eq-kf-model`, its position
measured at noise $\sigma_R = 0.5$ for 120 steps.

**Part a)** Write `kalman_step(x, P, z, F, Q, H, R)` returning
`(x, P, nu, S)`: predict by {eq}`eq-kf-predict`, then update by
{eq}`eq-kf-update`, with `.item()` to read the $1\times1$ innovation
variance. Initialise at $\mathbf{x}_0 = (z_0, 0)$, $P_0 =
\operatorname{diag}(\sigma_R^2, 1)$ and filter the remaining 119
measurements.

**Write this one yourself** — it is Exercise 2's step with a dynamics
matrix in front, and that refactoring is the notebook's thesis.

**Part b)** Gate the covariance exactly as in Exercise 2 (symmetric to
$10^{-13}$, positive definite throughout), and gate that filtering
*helps*: the filtered position's RMS error against the simulated truth
is smaller than the raw measurements' RMS error $\approx \sigma_R$ —
the filter is averaging along the trajectory, and it must beat the
sensor it reads.

**Part c)** Draw the track: truth, measurements, the filtered position
and its published $\pm2\sqrt{P_{11}}$ band — wide at the start, then
settling as the gain does.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    sym_kf < 1e-13 and min_eig_kf > 0.0,
    "the filter's covariance stays symmetric positive definite for 119 "
    "steps (Eqs. 5-6)",
    f"symmetry {sym_kf:.1e}, min eigenvalue {min_eig_kf:.2e} — the same "
    "health checks as RLS, surviving the dynamics",
)
validate.check(
    rms_filter < rms_meas,
    "and filtering beats the sensor it reads",
    f"RMS {rms_filter:.3f} against the measurements' {rms_meas:.3f}: the "
    "dynamics let every past measurement testify about the present, which "
    "is what the recursion is for",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4: Freeze the dynamics and the filter IS least squares

The claim behind the notebook's title, gated literally: run Exercise
3's `kalman_step` with the state frozen — $F = I$, $Q = 0$, unit
measurement noise $R = 1$, and the streaming rows as time-varying
measurement maps $H_k = \mathbf{a}_k^{\top}$ — and it must reproduce
Exercise 2's recursive least squares, step for step.

**Part a)** Stream the same 78 rows through `kalman_step` from the same
exact two-row start, and gate the frozen filter against the RLS
estimates at $10^{-12}$ relative at **every** step — with $F = I$ the
prediction does nothing, and {eq}`eq-kf-update` at $R = 1$ *is*
{eq}`eq-kf-sm`, symbol for symbol.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.below(
    worst_frozen, 1e-12,
    "Kalman with F = I, Q = 0, R = 1 reproduces recursive least squares "
    "step for step",
    "the title's claim as a measurement: the filter is least squares "
    "with a dynamics matrix in front, and freezing the dynamics removes "
    "the difference to rounding",
)

## Exercise 5: The gain settles where the Riccati equation says

Exercise 3's filter never referenced {eq}`eq-kf-dare`, yet its gain
must converge to the fixed point — computable in one call that filters
nothing. Two independent routes, one number.

**Part a)** Solve {eq}`eq-kf-dare` with
`scipy.linalg.solve_discrete_are(F.T, H.T, Q, [[R]])` and form the
steady gain $K_{\infty} = P_{\infty}H^{\top}/(HP_{\infty}H^{\top}+R)$.
Re-run the filter recording the gain at every step, and gate the final
gain against $K_{\infty}$ to $10^{-8}$ relative (measured: near
$10^{-15}$ — the Riccati iteration converges long before step 119).

**Part b)** Draw both gain components against the step with the DARE
values dashed: a dozen steps of transient, then the constants the
filter was always heading for.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.below(
    gap_gain, 1e-8,
    "the running filter's gain lands on the Riccati fixed point (Eq. 7)",
    "solve_discrete_are filtered nothing, the filter solved no Riccati "
    "equation, and they agree near rounding — two routes to one steady "
    "state",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6: The innovations testify

A correctly-modelled filter leaves nothing predictable behind: the
innovation sequence should be **white**, with the variance $S$ the
filter itself published. Both are statistics of a seeded run, so both
are gated in generous bands and reported precisely.

**Part a)** Normalise Exercise 3's innovations as $\nu_t/\sqrt{S_t}$
and gate the moments: standard deviation inside $[0.8, 1.25]$
(measured $1.06$ over 119 samples, where the standard error alone is
$\approx 0.065$), and lag-$1,2,3$ autocorrelations each inside
$\pm0.35$ (nearly four standard errors at this run length).

**Part b)** Draw the normalised innovations with the $\pm2$ band: a
white scatter, no drift, no ringing — the filter's self-diagnosis,
passed in plain sight.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    0.8 < std_nu < 1.25,
    "the innovations carry exactly the variance the filter published",
    f"std {std_nu:.3f} on 119 samples (standard error about 0.065): the "
    "filter's error bars are calibrated, not decorative",
)
validate.check(
    all(abs(a) < 0.35 for a in acorr),
    "and they are white: the filter left nothing predictable behind",
    f"lag-1,2,3 autocorrelations {[f'{a:.2f}' for a in acorr]} inside a "
    "generous +-0.35 band — a mis-modelled dynamics would ring here "
    "first",
)

In [ ]:
# (solution hidden on the public site)


---
## Notebook summary

**Streaming equals batch, at rounding, 78 times.** The rank-one
information update reproduced `np.linalg.lstsq` on every prefix with a
worst relative gap near $10^{-15}$ (gated at $10^{-11}$), and the
Sherman–Morrison gain form tracked both the information route
($10^{-12}$) and the explicit inverse ($10^{-11}$) without ever
inverting — the [§1.3](../01-matrices/inverses-rank-cr.ipynb) identity
doing production work at $O(p^2)$ per row.

**The filter is that update with dynamics in front.** Assembled from
{eq}`eq-kf-predict`–{eq}`eq-kf-update`, it tracked the
constant-velocity target with a covariance that stayed symmetric to
$10^{-16}$-scale and positive definite for all 119 steps, beat the raw
sensor's RMS error, and — with $F = I$, $Q = 0$, $R = 1$ — reproduced
recursive least squares to $10^{-15}$-scale, step for step: the title,
measured.

**Two Riccati routes, one gain.** The running filter's gain landed on
`solve_discrete_are`'s fixed point near rounding (gated at $10^{-8}$),
a dozen transient steps after starting from an ignorant $P_0$.

**The diagnostics passed in the open.** Normalised innovations came out
white — std $1.06$ against the published variance, lag-1,2,3
autocorrelations under $0.08$ in a $\pm0.35$ band — the one set of
statistical gates in the notebook, stated with their standard errors.

**Methods introduced.** The rank-one normal-equation stream,
`rls_gain_step` via Sherman–Morrison, exact two-row initialisation,
`kalman_step` (predict–update with `.item()` scalar hygiene), the
frozen-dynamics reduction, the DARE cross-check, and
innovation-whiteness testing with explicit bands.

## Outlook

- **Weights are covariances.** Replacing $1 + \mathbf{a}^{\top}P\mathbf{a}$
  by $R + \mathbf{a}^{\top}P\mathbf{a}$ is weighted least squares
  ([§2.3](../02-orthogonality/least-squares-four-ways.ipynb)) done
  recursively; a *forgetting factor* $\lambda < 1$ on $P$ turns RLS
  into a tracker of slowly drifting parameters — adaptive filtering's
  founding trick.
- **Square-root filters.** At $p$ in the thousands, $P - KHP$ can lose
  definiteness to rounding; propagating a Cholesky or QR factor of $P$
  instead ([§3.3](../03-eigenvalues/positive-definite-cholesky.ipynb),
  [§2.2](../02-orthogonality/gram-schmidt-qr.ipynb)) keeps the
  covariance a covariance by construction — the numerically serious
  form of everything built here.
- **Smoothing runs the recursion backwards.** The filter uses only the
  past; the Rauch–Tung–Striebel smoother adds a backward sweep that
  revises every estimate with the whole record — the same algebra, read
  in the other direction.
- **When the model bends.** Nonlinear dynamics linearised per step give
  the extended Kalman filter; sampled covariances give the unscented
  variant; and the innovations of Exercise 6 remain the diagnostic in
  every case — structure in that scatter is how a wrong model
  announces itself.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()